In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. 전처리 함수 정의
def preprocess_image(image, label):
    image = tf.image.resize(image, (150, 150)) # 이미지 크기 조정
    image = tf.cast(image, tf.float32) / 255.0 # 정규화(Rescaling)
    return image, label

# 2. 데이터셋 셔플 및 배치 설정
BATCH_SIZE = 32
train_batches = train_data.map(preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_batches = test_data.map(preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [6]:
# 3. 모델 설계
model = models.Sequential([
    # 첫 번째 Conv + Pooling (입력 크기 150x150x3)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D((2, 2)),

    # 두 번째 Conv + Pooling
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # 세 번째 Conv (특징 추출 강화)
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # 평탄화 및 분류기
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid') # 개(1) 또는 고양이(0) 이진 분류
])

# 4. 모델 컴파일
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 모델 구조 출력
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     2,367,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,460,865 (9.39 MB)

 Trainable params: 2,460,865 (9.39 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# 5. 모델 학습 (시간 관계상 10회만 진행)
history = model.fit(
    train_batches,
    epochs=10,
    validation_data=test_batches
)

# 6. 최종 정확도 평가
loss, accuracy = model.evaluate(test_batches)
print(f"\n최종 테스트 정확도: {accuracy:.4f}")

Epoch 1/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 934s 2s/step - accuracy: 0.6665 - loss: 0.5967 - val_accuracy: 0.7307 - val_loss: 0.5263
Epoch 2/10
 54/582 ━━━━━━━━━━━━━━━━━━━━ 13:04 1s/step - accuracy: 0.7611 - loss: 0.5089

KeyboardInterrupt: 